# CS224 Natural Language Understanding - Word Relatedness## 1. Word Relatedness / Word Similarity — OverviewWord relatedness and word similarity are standard benchmarks used to evaluate distributed word representations (vector space models). These tasks assess how well a model captures semantic relationships between words by comparing model-derived distances with human judgments.The evaluation data consist of CSV files containing word pairs, where each pair is associated with a human-annotated relatedness or similarity score. Similarity focuses on how alike two words are (e.g., car–automobile), whereas relatedness captures broader semantic association (e.g., car–road).Models are evaluated by computing vector distances or similarities (e.g., cosine similarity) between word embeddings and comparing these values to the human scores.

## 2. Setup and Imports

In [ ]:
# Install required packages (uncomment if needed)# !pip install numpy pandas scipy scikit-learn transformers

In [ ]:
# Import librariesfrom collections import defaultdictimport csvimport itertoolsimport numpy as npimport osimport pandas as pdimport randomfrom scipy.stats import spearmanrimport vsmimport utilsutils.fix_random_seeds()VSM_HOME = os.path.join('data', 'vsmdata')DATA_HOME = os.path.join('data', 'worrelatnedness')

## 3. Load Development Dataset

In [ ]:
dev_df = pd.read_csv(    os.path.join(DATA_HOME, "cs224-wordrelatedness-dev.csv"))print("Development dataset head:")print(dev_df.head())print(f"\nDataset shape: {dev_df.shape[0]}")

## 4. Vocabulary Analysis

In [ ]:
dev_vocab = set(dev_df.word1.values) | set(dev_df.word2.values)print(f"Development vocabulary size: {len(dev_vocab)}")task_index = pd.read_csv(    os.path.join(VSM_HOME, 'yelp_window-scaled.csv.gz'),    usecols=[0], index_col=0)full_task_vocab = list(task_index.index)print(f"Full task vocabulary size: {len(full_task_vocab)}")

## 5. Random Baseline

In [ ]:
random_baseline_df = pd.read_csv(    os.path.join(VSM_HOME, 'yelp_window-scaled.csv.gz'), index_col=0)random_pred_df, random_rho = vsm.word_relatedness_evaluation(    dev_df, random_baseline_df, distfunc=vsm.cosine)print(f"Random baseline Spearman correlation: {random_rho:.3f}")

## 6. Count-based Baseline

In [ ]:
count_baseline_df = pd.read_csv(    os.path.join(VSM_HOME, 'yelp_window-scaled.csv.gz'), index_col=0)count_pred_df, count_rho = vsm.word_relatedness_evaluation(    dev_df, count_baseline_df, distfunc=vsm.cosine)print(f"Count-based baseline Spearman correlation: {count_rho:.3f}")

## 7. Error Analysis

In [ ]:
def error_analysis(pred_df):    pred_df = pred_df.copy()    pred_df['relatedness_rank'] = _normalized_ranking(pred_df['score'])    pred_df['score_rank'] = _normalized_ranking(pred_df['predicted'])    pred_df['error'] = abs(pred_df['relatedness_rank'] - pred_df['score_rank'])    return pred_dfdef _normalized_ranking(series):    ranks = series.rank(method='dense')    return ranks / ranks.sum()print("Error analysis (head):")print(error_analysis(count_pred_df).head())print("\nError analysis (tail):")print(error_analysis(count_pred_df).tail())

## 8. PPMI Baseline

In [ ]:
def run_giga_ppmi_baseline():    giga_df = pd.read_csv(        os.path.join(VSM_HOME, 'giga_window20-flat.csv.gz'), index_col=0)    ppmi_df = vsm.ppmi(giga_df)    pred_df, rho = vsm.word_relatedness_evaluation(        dev_df, ppmi_df, distfunc=vsm.cosine)    return pred_df, rhodef test_run_giga_ppmi_baseline(func):    pred_df, rho = func()    rho = round(rho, 3)    expected = 0.351    print(f"PPMI baseline test: Expected rho={expected}, Got rho={rho}")test_run_giga_ppmi_baseline(run_giga_ppmi_baseline)

## 9. PPMI + LSA Pipeline

In [ ]:
def run_ppmi_lsa_pipeline(count_df, k):    ppmi_df = vsm.ppmi(count_df)    lsa_df = vsm.lsa(ppmi_df, k=k)    pred_df, rho = vsm.word_relatedness_evaluation(        dev_df, lsa_df, distfunc=vsm.cosine)    return pred_df, rhodef test_run_ppmi_lsa_pipeline(func):    giga20 = pd.read_csv(        os.path.join(VSM_HOME, "giga_window20-flat.csv.gz"), index_col=0)    pred_df, rho = func(giga20, k=10)    rho = round(rho, 3)    expected = 0.319    print(f"PPMI+LSA pipeline test: Expected rho={expected}, Got rho={rho}")test_run_ppmi_lsa_pipeline(run_ppmi_lsa_pipeline)

## 10. T-test Reweighting

In [ ]:
def ttest(df):    col_means = df.mean(axis=0)    col_stds = df.std(axis=0)    result = df.copy()    for col in df.columns:        if col_stds[col] != 0:            result[col] = (df[col] - col_means[col]) / col_stds[col]        else:            result[col] = 0    return resultdef test_ttest_implementation(func):    X = pd.DataFrame([        [1., 4., 3., 0.],        [2., 4., 7., 8.]    ])    actual = np.array([        [-0.70711, 0.0, -0.70711, -0.70711],        [0.70711, 0.0, 0.70711, 0.70711]    ])    predicted = func(X)    print(f"t-test result:\n{predicted.round(5)}")test_ttest_implementation(ttest)

## 11. Pooled BERT Representations**Note:** Requires `transformers` library. Uncomment the code below to run.

In [ ]:
# from transformers import BertModel, BertTokenizer# def evaluate_pooled_bert(rel_df, layer, pool_func):#     bert_weights_name = 'bert-base-uncased'#     bert_tokenizer = BertTokenizer.from_pretrained(bert_weights_name)#     bert_model = BertModel.from_pretrained(bert_weights_name)#     vocab = set(rel_df.word1.values) | set(rel_df.word2.values)#     vsm_df = vsm.create_subword_pooling_vsm(#         vocab, bert_model, bert_tokenizer, layer, pool_func)#     return vsm.word_relatedness_evaluation(rel_df, vsm_df)# def test_evaluate_pooled_bert(func):#     rel_df = pd.DataFrame([#         {'word1': 'porcupine', 'word2': 'capybara', 'score': 0.6},#         {'word1': 'antelope', 'word2': 'book', 'score': 0.5}#     ])#     layer = 2#     pool_func = vsm.max_pooling#     pred_df, rho = func(rel_df, layer, pool_func)#     rho = round(rho, 2)#     expected_rho = 0.40#     print(f"Pooled BERT test: Expected rho={expected_rho}, Got rho={rho}")# test_evaluate_pooled_bert(evaluate_pooled_bert)

## 12. Learned Distance Functions (KNN)

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.neighbors import KNeighborsRegressordef run_knn_score_model(vsm_df, dev_df, test_size=0.20):    X = knn_feature_matrix(vsm_df, dev_df)    y = dev_df['score'].values    X_train, X_test, y_train, y_test = train_test_split(        X, y, test_size=test_size, random_state=42)    model = KNeighborsRegressor()    model.fit(X_train, y_train)    return model.score(X_test, y_test)def knn_feature_matrix(vsm_df, rel_df):    features = []    for _, row in rel_df.iterrows():        feat = knn_represent(row['word1'], row['word2'], vsm_df)        features.append(feat)    return np.array(features)def knn_represent(word1, word2, vsm_df):    v1 = vsm_df.loc[word1].values    v2 = vsm_df.loc[word2].values    return np.concatenate([v1, v2])

## 13. Testing KNN Functions

In [ ]:
def test_knn_feature_matrix(func):    rel_df = pd.DataFrame([        {'word1': 'w1', 'word2': 'w2', 'score': 0.1},        {'word1': 'w1', 'word2': 'w3', 'score': 0.2}    ])    vsm_df = pd.DataFrame([        [1, 2, 3.],        [4, 5, 6.],        [7, 8, 9.]    ], index=['w1', 'w2', 'w3'])    expected = np.array([        [1, 2, 3, 4, 5, 6.],        [1, 2, 3, 7, 8, 9.]    ])    result = func(vsm_df, rel_df)    assert np.array_equal(result, expected), \        f"Your 'knn_feature_matrix' returns:\n{result}\nWe expect:\n{expected}"    print("knn_feature_matrix test passed!")def test_knn_represent(func):    vsm_df = pd.DataFrame([        [1, 2, 3.],        [4, 5, 6.],        [7, 8, 9.]    ], index=['w1', 'w2', 'w3'])    result = func('w1', 'w2', vsm_df)    expected = np.array([1, 2, 3, 4, 5, 6.])    assert np.array_equal(result, expected), \        f"Your knn_represent returns:\n{result}\nWe expect:\n{expected}"    print("knn_represent test passed!")test_knn_represent(knn_represent)test_knn_feature_matrix(knn_feature_matrix)print("\n" + "="*50)print("All tests completed!")print("="*50)